# Production Architecture: Real-Time English (ASL) Sign Language Recognition

### What changed from baseline
- **`mp.solutions.hands` only, `max_num_hands=1`** — face/pose detection removed entirely
- **Threaded camera** with drop-queue — always the latest frame, no buffer lag
- **`@tf.function` compiled inference** — graph-mode, no Python overhead per prediction
- **`model(x, training=False)`** — bypasses `predict()` batch pipeline
- **Raised MediaPipe thresholds** (`detection=0.70`, `tracking=0.65`)
- **Stabilisation params kept at 15/11/0.75/1.2** — production-tuned values


## 0. Imports

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import tensorflow as tf
import time
import threading
import queue
from collections import deque
import os

print("✓ Libraries imported successfully.")

## 1. GPU Configuration

In [ ]:
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✓ GPU configured: {len(gpus)} GPU(s)")
    except RuntimeError as e:
        print(f"⚠ GPU config error: {e}")
else:
    print("ℹ No GPU detected, using CPU")

## 2. Configuration & Parameters

`STABILIZATION_WINDOW_SIZE`, `STABILIZATION_THRESHOLD`, `MIN_CONFIDENCE`, and `HOLD_COOLDOWN_SECONDS`
are kept at production-tuned values. MediaPipe thresholds raised from original `0.5/0.5`.


In [ ]:
#MODEL_PATH = r"asl_mediapipe_mlp_model.h5"
MODEL_PATH = r"asl_mediapipe_mlp_model_engineered.h5"

STABILIZATION_WINDOW_SIZE  = 15     # rolling-buffer length
STABILIZATION_THRESHOLD    = 11     # 11/15 ≈ 73% majority required
MIN_CONFIDENCE             = 0.75   # predictions below this are discarded
HOLD_COOLDOWN_SECONDS      = 1.2    # lock-out after a commit

MP_DETECTION_CONFIDENCE    = 0.70   # raised from 0.5
MP_TRACKING_CONFIDENCE     = 0.65   # raised from 0.5

DISPLAY_WIDTH  = 1280
DISPLAY_HEIGHT = 720

CLASS_LABELS = [
    "A","B","C","D","E","F","G","H",
    "I","J","K","L","M","N","O","P",
    "Q","R","S","T","U","V","W","X",
    "Y","Z","del","nothing","space"
]
print("✓ Settings configured")

## 3. Threaded Camera

A daemon thread continuously pushes the newest frame into a **drop-queue of size 1**.
The main loop always gets the most recent frame — zero accumulation, minimal latency.


In [ ]:
class ThreadedCamera:
    """Background daemon thread keeps queue fresh (size 1 = always latest frame)."""
    def __init__(self, src=0):
        self.cap = cv2.VideoCapture(src)
        self.cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)   # flush OS-level ring buffer
        self.cap.set(cv2.CAP_PROP_FRAME_WIDTH,  DISPLAY_WIDTH)
        self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, DISPLAY_HEIGHT)
        self.cap.set(cv2.CAP_PROP_FPS, 30)
        self._q      = queue.Queue(maxsize=1)
        self._active = True
        self._thread = threading.Thread(target=self._reader, daemon=True)
        self._thread.start()

    def _reader(self):
        while self._active:
            ret, frame = self.cap.read()
            if not ret:
                continue
            if not self._q.empty():
                try:
                    self._q.get_nowait()
                except queue.Empty:
                    pass
            self._q.put(frame)

    def read(self):
        return self._q.get()

    def is_opened(self):
        return self.cap.isOpened()

    def release(self):
        self._active = False
        self._thread.join(timeout=2)
        self.cap.release()

print("✓ ThreadedCamera defined")

## 4. MediaPipe Hands (single hand)

`max_num_hands=1` — no face/pose processing. 21 pts × 3 coords = 63 features, same layout as training CSV.


In [ ]:
import mediapipe as mp
mp_hands   = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode        = False,
    max_num_hands            = 1,
    min_detection_confidence = MP_DETECTION_CONFIDENCE,
    min_tracking_confidence  = MP_TRACKING_CONFIDENCE,
)

# ── Feature Engineering (must match training) ──

# ============================================
# FEATURE ENGINEERING FUNCTION
# Shared between Training and Production
# ============================================

def compute_angle(a, b, c):
    """Compute angle (radians) at joint b, formed by points a-b-c."""
    ba = a - b
    bc = c - b
    cosine = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    cosine = np.clip(cosine, -1.0, 1.0)
    return np.arccos(cosine)

# Joint triplets for angle computation (15 angles)
# Each tuple: (parent, joint, child) -- angle is measured at 'joint'
ANGLE_TRIPLETS = [
    # Thumb  (3 angles)
    (0, 1, 2), (1, 2, 3), (2, 3, 4),
    # Index  (3 angles)
    (0, 5, 6), (5, 6, 7), (6, 7, 8),
    # Middle (3 angles)
    (0, 9, 10), (9, 10, 11), (10, 11, 12),
    # Ring   (3 angles)
    (0, 13, 14), (13, 14, 15), (14, 15, 16),
    # Pinky  (3 angles)
    (0, 17, 18), (17, 18, 19), (18, 19, 20),
]

def extract_engineered_features(landmarks_array):
    """
    Takes a (21, 3) numpy array of hand landmarks.
    Returns a (78,) feature vector:
      - 63 wrist-relative coordinates (21 pts x 3)
      - 15 joint angles (3 per finger)
    """
    # 1. Wrist-relative normalization
    wrist = landmarks_array[0]  # (3,)
    relative = landmarks_array - wrist  # (21, 3)
    relative_flat = relative.flatten()  # (63,)

    # 2. Joint angles
    angles = []
    for a_idx, b_idx, c_idx in ANGLE_TRIPLETS:
        angle = compute_angle(
            landmarks_array[a_idx],
            landmarks_array[b_idx],
            landmarks_array[c_idx]
        )
        angles.append(angle)
    angles = np.array(angles, dtype=np.float32)  # (15,)

    # 3. Concatenate
    return np.concatenate([relative_flat, angles])  # (78,)

NUM_ENGINEERED_FEATURES = 78
print(f"\u2705 Feature engineering function defined: {NUM_ENGINEERED_FEATURES} features per hand")
print(f"   - 63 wrist-relative coordinates (21 landmarks \u00d7 3 axes)")
print(f"   - 15 joint bend angles (3 per finger)")

def extract_features(results):
    """Returns (1, 78) float32 or None. First detected hand only."""
    if not results.multi_hand_landmarks:
        return None
    lm = results.multi_hand_landmarks[0].landmark
    landmarks_raw = np.array([[p.x, p.y, p.z] for p in lm], dtype=np.float32)
    features = extract_engineered_features(landmarks_raw)
    return features.reshape(1, -1)

print("\u2713 MediaPipe Hands configured with engineered features (78-dim)")

## 5. Stabilisation Tracker

`commit-once-then-wait` logic. **Fix added**: if a *different* sign appears during cooldown,
the lock releases immediately so the new letter can start stabilising without waiting out the full timer.


In [ ]:
class StabilizationTracker:
    def __init__(self):
        self.buffer          = deque(maxlen=STABILIZATION_WINDOW_SIZE)
        self.committed_label = None
        self.cooldown_until  = 0.0

    def update(self, label, confidence):
        now = time.time()
        if label is None or label == "nothing":
            self.buffer.clear()
            self.committed_label = None
            return None, 0.0, "waiting", 0
        if self.committed_label == label and now < self.cooldown_until:
            return label, confidence, "cooldown", 100
        if self.committed_label is not None and self.committed_label != label:
            self.committed_label = None   # unlock: different letter appeared
        self.buffer.append(label)
        count    = self.buffer.count(label)
        progress = int(min(100, (count / STABILIZATION_THRESHOLD) * 100))
        if count >= STABILIZATION_THRESHOLD and len(self.buffer) == STABILIZATION_WINDOW_SIZE:
            self.committed_label = label
            self.cooldown_until  = now + HOLD_COOLDOWN_SECONDS
            self.buffer.clear()
            return label, confidence, "committed", 100
        return label, confidence, "stabilizing", progress

print("✓ StabilizationTracker defined")

## 6. Load Model & Compile Inference

`@tf.function` traces the graph once. Warm-up call eliminates JIT delay on frame 1.


In [ ]:
model = None
try:
    if os.path.exists(MODEL_PATH):
        model = tf.keras.models.load_model(MODEL_PATH)
        print("✓ ASL MLP model loaded")
    else:
        print(f"❌ Model not found at {MODEL_PATH}")
except Exception as e:
    print(f"❌ Error loading model: {e}")

if model is not None:
    @tf.function(input_signature=[tf.TensorSpec(shape=[1, 78], dtype=tf.float32)])
    def run_inference(x):
        return model(x, training=False)

    _ = run_inference(tf.zeros([1, 78], dtype=tf.float32))  # warm-up
    print("✓ Inference compiled and warmed up")

## 7. Main Recognition Loop

In [ ]:
def run_sign_recognition():
    if model is None:
        print("❌ Model not loaded.")
        return
    cam = ThreadedCamera(src=0)
    if not cam.is_opened():
        print("❌ Cannot access camera")
        return
    print("\n================================================")
    print("🤟 ASL SIGN LANGUAGE RECOGNITION — PRODUCTION")
    print("================================================")
    print("  q = quit    c = clear sentence")
    print("================================================\n")
    window_name = "ASL SLR — Production"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, DISPLAY_WIDTH, DISPLAY_HEIGHT)
    tracker            = StabilizationTracker()
    predicted_sentence = ""
    fps_start          = time.time()
    frame_count        = 0
    fps_display        = 0
    try:
        while True:
            frame = cam.read()
            rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            rgb.flags.writeable = False
            results = hands.process(rgb)
            rgb.flags.writeable = True
            display_status = "No hand detected"
            status_color   = (150, 150, 150)
            if results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(
                    frame, results.multi_hand_landmarks[0], mp_hands.HAND_CONNECTIONS)
            features = extract_features(results)
            if features is not None:
                prediction = run_inference(tf.constant(features))[0].numpy()
                class_idx  = int(np.argmax(prediction))
                conf       = float(prediction[class_idx])
                raw_label  = CLASS_LABELS[class_idx]
                if conf < MIN_CONFIDENCE:
                    display_status = f"Low confidence: {conf:.0%}"
                    status_color   = (0, 100, 255)
                    tracker.update(None, 0.0)
                else:
                    tracked_label, tracked_conf, status, progress = tracker.update(raw_label, conf)
                    if status == "stabilizing":
                        display_status = f"{tracked_label} ({tracked_conf:.0%})  {progress}%"
                        status_color   = (0, 255, 255)
                    elif status == "cooldown":
                        display_status = f"{tracked_label} ({tracked_conf:.0%})  ✓ Committed"
                        status_color   = (255, 200, 0)
                    elif status == "committed":
                        display_status = f"{tracked_label} ({tracked_conf:.0%})  ✓ COMMITTED!"
                        status_color   = (0, 255, 0)
                        if tracked_label == "space":
                            if not predicted_sentence.endswith(" "):
                                predicted_sentence += " "
                        elif tracked_label == "del":
                            predicted_sentence = predicted_sentence[:-1]
                        elif tracked_label != "nothing":
                            predicted_sentence += tracked_label
            else:
                tracker.update(None, 0.0)
            frame = cv2.flip(frame, 1)
            frame_count += 1
            if time.time() - fps_start >= 1.0:
                fps_display = frame_count
                frame_count = 0
                fps_start   = time.time()
            cv2.rectangle(frame, (0, 0), (DISPLAY_WIDTH, 70), (30, 30, 30), -1)
            cv2.putText(frame, f"FPS: {fps_display}  |  q=quit  c=clear",
                        (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
            cv2.putText(frame, display_status,
                        (10, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.9, status_color, 2)
            bar_h = 80
            cv2.rectangle(frame,
                          (0, DISPLAY_HEIGHT - bar_h), (DISPLAY_WIDTH, DISPLAY_HEIGHT),
                          (30, 30, 30), -1)
            cv2.putText(frame,
                        predicted_sentence[-40:] if predicted_sentence else "_",
                        (20, DISPLAY_HEIGHT - 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.4, (255, 255, 255), 2)
            cv2.imshow(window_name, frame)
            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break
            elif key == ord("c"):
                predicted_sentence = ""
                tracker.update(None, 0.0)
                print("🗑 Sentence cleared")
    finally:
        cam.release()
        cv2.destroyAllWindows()
        print(f"\n📝 Final sentence: {predicted_sentence}")


run_sign_recognition()